In [ ]:
from dotenv import load_dotenv
from llama_cloud_services import LlamaExtract


# Load environment variables (put LLAMA_CLOUD_API_KEY in your .env file)
load_dotenv(override=True)

: 

In [ ]:
import os
from llama_index.llms.gemini import Gemini

llm = Gemini(
    model="models/gemini-2.0-flash",
    api_key=os.getenv("GEMINI_API_KEY"),
)

In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()
from llama_parse import LlamaParse
from llama_index.core.node_parser import MarkdownElementNodeParser


pdf_file = "070znnle73289_s1097.pdf"
documents = LlamaParse(
    result_type="markdown",
    auto_mode=True,
    auto_mode_trigger_on_table_in_page=True,
).load_data(pdf_file)

In [ ]:
from copy import deepcopy
from llama_index.core.schema import TextNode
from llama_index.core import VectorStoreIndex


def get_page_nodes(docs, separator="\n---\n"):
    """Split each document into page node, by separator."""
    nodes = []
    for doc in docs:
        doc_chunks = doc.text.split(separator)
        for doc_chunk in doc_chunks:
            node = TextNode(
                text=doc_chunk,
                metadata=deepcopy(doc.metadata),
            )
            nodes.append(node)

    return nodes

In [ ]:
page_nodes = get_page_nodes(documents)
print(type(page_nodes[0].get_content()))
content = page_nodes[0].get_content()
print(content)

In [ ]:
from llama_index.core.program import LLMTextCompletionProgram

from pydantic import BaseModel, Field
from typing import List, Optional

# Defining the data schema
class Item(BaseModel):
    item_no: str = Field(description="The number of the item.It should be a number.It is not provided should be empty")
    description: str = Field(description="The description of the item")
    manufacturer: str = Field(description="The manufacturer of the item")
    identification_data: str = Field(description="The identification data of the item.")
    quantity: str = Field(description="The quantity of the item.")
    designation: str = Field(description="The designation on the diagramm or designacion en el plano of the item")
    sheet: str = Field(description="The sheet line diagramm or hoja de plano of the item")
    location: str = Field(description="The location of the item")


class ListOfItems(BaseModel):
    items: List[Item] = Field(description="The list of items")
    
prompt_template_str = """\
From the following information, create a list of items:
{content} \
"""
program = LLMTextCompletionProgram.from_defaults(
    llm=llm,
    output_cls=ListOfItems,
    prompt_template_str=prompt_template_str,
    verbose=True,
)


output = program(content=content)

In [ ]:
import pandas as pd
# Convert the Pydantic models to dictionaries
dict_list_of_items = output.model_dump()

# Create a DataFrame from the dictionaries
df = pd.DataFrame(dict_list_of_items["items"])
df